In [1]:
import os
import cv2
import numpy as np
import torch
import timm
from tqdm import tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [3]:
IMG_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\data\raw\processed\images"
LBL_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\data\raw\processed\labels"

OUTPUT_DIR = r"C:\Users\PRAVEEN S\Desktop\BrainTumor-Cloud-AI\models"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [4]:
model = timm.create_model(
    "swin_base_patch4_window7_224",  # Swin-B
    pretrained=True,
    num_classes=0
)


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

c:\Users\PRAVEEN S\AppData\Local\Programs\Python\Python39\lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PRAVEEN S\.cache\huggingface\hub\models--timm--swin_base_patch4_window7_224.ms_in22k_ft_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [5]:
def preprocess_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    img = img.astype(np.float32) / 255.0
    img = torch.tensor(img).permute(2, 0, 1)
    return img

In [6]:
image_files = sorted(os.listdir(IMG_DIR))

features = []
labels = []

with torch.no_grad():
    for img_name in tqdm(image_files, desc="Extracting Swin features"):
        img_path = os.path.join(IMG_DIR, img_name)
        lbl_path = os.path.join(LBL_DIR, img_name.replace(".png", ".txt"))

        img = preprocess_image(img_path)
        img = img.unsqueeze(0).to(device)

        feat = model(img)             # shape: (1, feature_dim)
        feat = feat.squeeze().cpu().numpy()

        with open(lbl_path) as f:
            label = int(f.read().strip())

        features.append(feat)
        labels.append(label)


Extracting Swin features: 100%|██████████| 6939/6939 [1:15:27<00:00,  1.53it/s]


In [7]:
features = np.array(features)
labels = np.array(labels)

print("Swin feature shape:", features.shape)
print("Labels shape:", labels.shape)


Swin feature shape: (6939, 1024)
Labels shape: (6939,)


In [9]:
np.save(os.path.join(OUTPUT_DIR, "swinB_features.npy"), features)

print("Swin features saved ✅")

Swin features saved ✅
